# 第 31 章：短期利率模型

本章每一个算例，全部重新算一遍。参数取自书中，而每一个数字都与书里印出的数值并排打印，
因此任何数字一旦偏离书中的值，都能一眼看出，而不会悄悄出错。

| 小节 | 复现的内容 | 例 |
|---|---|---|
| 1 | Vasicek 由三个起始利率给出的曲线 | 31.1 |
| 2 | 匹配波动率下 CIR 与 Vasicek 的比较 | 31.2 |
| 3 | $a$ 与 $\sigma$ 各自控制什么 | 31.3 |
| 4 | 拟合到一条曲线的 Hull–White 方法 | 第 31.6 节 |
| 5 | 上限期权与互换期权的闭式解 | 31.4 |
| 6 | 同一份上限单元的三种算法 | 31.5 |
| 7 | 远期价格不是 $Q$ 下的期望 | 31.6 |
| 8 | 一步一步把树建起来 | 31.7、第 31.8 节 |
| 9 | 一份有五个日期的期权 | 31.8 |
| 10 | 每一份欧式都匹配，百慕大不匹配 | 31.9 |
| 11 | Z 利差与期权调整利差 | 31.10 |
| 12 | 完全相关的代价 | 31.11 |
| 13 | 模型说利率低于零的频率 | 31.12 |

这里几乎所有计算都是确定性的，都精确重现。唯一的一次模拟在第 6 节，它从一个有闭式解的分布中
抽样，因此带标准误差、但不带任何离散化偏差。整个笔记本大约运行二十秒。

In [1]:
import numpy as np
from math import ceil, exp, log, pi, sqrt
from scipy.optimize import brentq
from scipy.stats import norm

N = norm.cdf


def show(label, got, printed, tol=6e-5):
    """把算出的值与书中印出的值并排打印。"""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %14.6f   book %13.6f   %s" % (label, got, printed, flag))

## 1. Vasicek 由三个起始利率给出的曲线：例 31.1

三个起始短期利率，一组参数，以及它们产生的曲线。三条都收敛到同一个长期利率，
而这个利率位于 $b$ *之下*：$-\sigma^2/(2a^2)$ 是一个凸性项，
它也是均衡模型的形状并非自由选择的第一个迹象。

In [2]:
A_V, B_V, SIG_V = 0.10, 0.05, 0.01


def vasicek_zero(r0, T, a=A_V, b=B_V, sigma=SIG_V):
    """Vasicek 的连续复利零息利率，由仿射债券价格得出。"""
    T = np.asarray(T, dtype=float)
    B = (1.0 - np.exp(-a * T)) / a
    lnA = (B - T) * (a * a * b - 0.5 * sigma ** 2) / a ** 2 - sigma ** 2 * B * B / (4.0 * a)
    return (B * r0 - lnA) / T


MATS = (0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0)
BOOK_V = ((2.073, 5.000, 7.926), (2.144, 4.998, 7.853), (2.275, 4.994, 7.713),
          (2.610, 4.971, 7.332), (3.020, 4.916, 6.812), (3.513, 4.810, 6.107),
          (3.783, 4.734, 5.684))
print("%9s %9s %9s %9s     %s" % ("maturity", "r0 = 2%", "r0 = 5%", "r0 = 8%", "book"))
for T, want in zip(MATS, BOOK_V):
    got = [100 * float(vasicek_zero(r0, T)) for r0 in (0.02, 0.05, 0.08)]
    flag = "ok" if max(abs(g - w) for g, w in zip(got, want)) < 5e-4 + 1e-9 else "DIFFERS"
    print("%8.1fy %8.3f%% %8.3f%% %8.3f%%     %s" % (T, got[0], got[1], got[2], flag))
show("R_infinity = b - sigma^2/(2a^2), %",
     100 * (B_V - SIG_V ** 2 / (2 * A_V ** 2)), 4.50, tol=5e-4)

 maturity   r0 = 2%   r0 = 5%   r0 = 8%     book
     0.5y    2.073%    5.000%    7.926%     ok
     1.0y    2.144%    4.998%    7.853%     ok
     2.0y    2.275%    4.994%    7.713%     ok
     5.0y    2.610%    4.971%    7.332%     ok
    10.0y    3.020%    4.916%    6.812%     ok
    20.0y    3.513%    4.810%    6.107%     ok
    30.0y    3.783%    4.734%    5.684%     ok
R_infinity = b - sigma^2/(2a^2), %                   4.500000   book      4.500000   ok


### 第三种形状：勉强存在

Vasicek 能产生驼峰形曲线，但起始利率必须落在一条仅 75 个基点宽的带内，
而这条带内任何地方能产生的最大驼峰约为十个基点。这两件事值得算出来而不是断言，
因为那条带就是“这个模型能不能做出驼峰形曲线”的全部答案。

下面的扫描把驼峰定义为峰相对两端中*较高*者的高度，一直算到六百年。
一条只是升到平台的曲线不算驼峰。

In [3]:
HGRID = np.arange(1, 12001) * 0.05          # 一直算到 600 年：驼峰必须是一个峰


def hump(zero, r0):
    """峰相对两端中较高者的高度，以基点计，以及它所在的期限。"""
    y = zero(r0, HGRID)
    k = int(np.argmax(y))
    if k in (0, len(y) - 1):
        return 0.0, None
    far = 2 * y[-1] - y[len(y) // 2 - 1]    # 用 600 年与 300 年两点按 1/T 外推到 R_infinity
    return 10000 * (y[k] - max(r0, far)), HGRID[k]


def largest_hump(zero, lo, hi):
    """先粗扫驼峰带，再在最佳起始利率附近细化。"""
    best = lo
    for step in (5e-5, 5e-7):
        grid = np.arange(lo, hi, step)
        best = max(grid, key=lambda r: hump(zero, r)[0])
        lo, hi = best - step, best + step
    h, peak = hump(zero, best)
    return h, best, peak


V_LO = B_V - 3 * SIG_V ** 2 / (4 * A_V ** 2)
show("humped band, lower edge, %", 100 * V_LO, 4.25, tol=5e-4)
show("humped band, width, bp", 10000 * (B_V - V_LO), 75.0, tol=5e-4)
h_v, r_v, t_v = largest_hump(vasicek_zero, V_LO, B_V)
show("largest hump Vasicek can make, bp", h_v, 10.0, tol=3e-2)
show("the r0 that produces it, %", 100 * r_v, 4.50, tol=1e-3)
show("and that r0 is R_infinity, %", 100 * r_v,
     100 * (B_V - SIG_V ** 2 / (2 * A_V ** 2)), tol=1e-3)
show("maturity of the peak, years", t_v, 12.5, tol=1e-2)

humped band, lower edge, %                           4.250000   book      4.250000   ok
humped band, width, bp                              75.000000   book     75.000000   ok
largest hump Vasicek can make, bp                   10.181604   book     10.000000   ok
the r0 that produces it, %                           4.500000   book      4.500000   ok
and that r0 is R_infinity, %                         4.500000   book      4.500000   ok
maturity of the peak, years                         12.550000   book     12.500000   ok


## 2. 匹配波动率下 CIR 与 Vasicek 的比较：例 31.2

两个模型只有在波动率可比时才可比。Vasicek 的扩散项是 $\sigma$，CIR 的是 $\sigma\sqrt{r}$，
因此当 CIR 的 $\sigma$ 取 $0.01/\sqrt{0.05} = 4.47\%$ 时，两者在 $r = b$ 处一致。
此时 Feller 条件远远满足。

第二张表是两条曲线之差。这就是保证利率为正的代价，而对短期限几乎为零。

In [4]:
SIG_CIR = 0.0447
GAMMA = sqrt(A_V ** 2 + 2 * SIG_CIR ** 2)


def cir_zero(r0, tau, a=A_V, b=B_V, sigma=SIG_CIR):
    """CIR 的零息利率。同样的仿射形态，不同的 B 与 A。"""
    tau = np.asarray(tau, dtype=float)
    g = sqrt(a * a + 2 * sigma * sigma)
    e = np.exp(g * tau)
    den = (g + a) * (e - 1.0) + 2.0 * g
    B = 2.0 * (e - 1.0) / den
    lnA = (2.0 * a * b / sigma ** 2) * (log(2.0 * g) + (a + g) * tau / 2.0 - np.log(den))
    return (B * r0 - lnA) / tau


show("sigma_CIR matching 1% at r = b", 0.01 / sqrt(B_V), 0.0447, tol=5e-4)
show("Feller: 2ab", 2 * A_V * B_V, 0.0100, tol=5e-4)
show("Feller: sigma^2", SIG_CIR ** 2, 0.0020, tol=2e-3)

BOOK_C = ((2.074, 5.000, 7.926), (2.144, 4.998, 7.852), (2.278, 4.994, 7.710),
          (2.626, 4.971, 7.317), (3.059, 4.918, 6.777), (3.586, 4.821, 6.057),
          (3.869, 4.757, 5.645))
print()
print("%9s %9s %9s %9s     %s" % ("maturity", "r0 = 2%", "r0 = 5%", "r0 = 8%", "book"))
for T, want in zip(MATS, BOOK_C):
    got = [100 * float(cir_zero(r0, T)) for r0 in (0.02, 0.05, 0.08)]
    flag = "ok" if max(abs(g - w) for g, w in zip(got, want)) < 5e-4 + 1e-9 else "DIFFERS"
    print("%8.1fy %8.3f%% %8.3f%% %8.3f%%     %s" % (T, got[0], got[1], got[2], flag))

print()
print("%9s %9s %9s %9s" % ("maturity", "r0 = 2%", "r0 = 5%", "r0 = 8%"))
for T in (1, 2, 5, 10, 20, 30):
    print("%8dy %9.1f %9.1f %9.1f"
          % (T, 10000 * float(cir_zero(0.02, T) - vasicek_zero(0.02, T)),
             10000 * float(cir_zero(0.05, T) - vasicek_zero(0.05, T)),
             10000 * float(cir_zero(0.08, T) - vasicek_zero(0.08, T))))
print("CIR 减 Vasicek，以基点计")

sigma_CIR matching 1% at r = b                       0.044721   book      0.044700   ok
Feller: 2ab                                          0.010000   book      0.010000   ok
Feller: sigma^2                                      0.001998   book      0.002000   ok

 maturity   r0 = 2%   r0 = 5%   r0 = 8%     book
     0.5y    2.074%    5.000%    7.926%     ok
     1.0y    2.144%    4.998%    7.852%     ok
     2.0y    2.278%    4.994%    7.710%     ok
     5.0y    2.626%    4.971%    7.317%     ok
    10.0y    3.059%    4.918%    6.777%     ok
    20.0y    3.586%    4.821%    6.057%     ok
    30.0y    3.869%    4.757%    5.645%     ok

 maturity   r0 = 2%   r0 = 5%   r0 = 8%
       1y       0.1       0.0      -0.1
       2y       0.3       0.0      -0.3
       5y       1.5       0.0      -1.5
      10y       4.0       0.2      -3.6
      20y       7.3       1.2      -5.0
      30y       8.6       2.3      -3.9
CIR 减 Vasicek，以基点计


### CIR 的驼峰

同样的扫描，用在 CIR 上。它的带比 Vasicek 的更窄，它的最大驼峰也更小，而这正是要点：
保证利率为正的约束，也压缩了曲线可能的形状。

CIR 的带没有简单的闭式解，因此它的下边缘是对驼峰高度求根得到的，而不是写下来的。

In [5]:
show("CIR's long rate 2ab/(gamma + a), %", 100 * 2 * A_V * B_V / (GAMMA + A_V), 4.58, tol=5e-4)
C_LO = brentq(lambda r: hump(cir_zero, r)[0] - 1e-9, 0.040, 0.0499, xtol=1e-9)
show("CIR's humped band, lower edge, %", 100 * C_LO, 4.40, tol=1e-3)
show("CIR's humped band, width, bp", 10000 * (B_V - C_LO), 60.1, tol=2e-3)
show("narrower than Vasicek's by, bp",
     10000 * (B_V - V_LO) - 10000 * (B_V - C_LO), 14.9, tol=1e-2)
h_c, r_c, t_c = largest_hump(cir_zero, C_LO, B_V)
show("largest hump CIR can make, bp", h_c, 8.0, tol=3e-2)
show("the r0 that produces it, %", 100 * r_c, 4.58, tol=1e-3)
show("maturity of the peak, years", t_c, 11.1, tol=2e-2)

CIR's long rate 2ab/(gamma + a), %                   4.580738   book      4.580000   ok
CIR's humped band, lower edge, %                     4.398788   book      4.400000   ok
CIR's humped band, width, bp                        60.121224   book     60.100000   ok
narrower than Vasicek's by, bp                      14.878776   book     14.900000   ok
largest hump CIR can make, bp                        7.791863   book      8.000000   ok
the r0 that produces it, %                           4.580738   book      4.580000   ok
maturity of the peak, years                         11.150000   book     11.100000   ok


## 3. $a$ 与 $\sigma$ 各自控制什么：例 31.3

Hull–White 下远期利率的波动率是 $\sigma e^{-aT}$，两个参数分工明确：$\sigma$ 设定水平，
$a$ 设定形状。在 $a = 0.01$ 下，三十年远期利率的波动率是一年期的四分之三；
在 $a = 0.20$ 下，它基本上没有任何波动率。

In [6]:
A_HW, SIG_HW, L = 0.08, 0.01, 100.0

print("%9s %9s %9s %9s %9s" % ("maturity", "a = 0.01", "a = 0.05", "a = 0.10", "a = 0.20"))
for T in (1, 2, 5, 10, 20, 30):
    print("%8dy %9.1f %9.1f %9.1f %9.1f"
          % (T, *(100.0 * exp(-a * T) for a in (0.01, 0.05, 0.10, 0.20))))
print("远期利率波动率，以基点计，取 sigma = 1%")

 maturity  a = 0.01  a = 0.05  a = 0.10  a = 0.20
       1y      99.0      95.1      90.5      81.9
       2y      98.0      90.5      81.9      67.0
       5y      95.1      77.9      60.7      36.8
      10y      90.5      60.7      36.8      13.5
      20y      81.9      36.8      13.5       1.8
      30y      74.1      22.3       5.0       0.2
远期利率波动率，以基点计，取 sigma = 1%


## 4. 拟合到一条曲线的 Hull–White 方法

从这里往后的每一步都需要 $A(t,T)$，而 $A(t,T)$ 取决于模型被拟合到哪条曲线。
本章出现了两条曲线：例 31.4 的 4\% 平坦曲线，以及第 31.7 节起使用的市场曲线
$R(0,T) = 4.2\% - 1.2\%e^{-0.25T}$。下面用一个小类把一条曲线，连同由它得出的
$\ln A(t,T)$、$P(t,T)$ 与 $\theta(t)$ 封装在一起，这样两者都不必写两遍。

In [7]:
def B_hw(t, T, a=A_HW):
    return (1.0 - np.exp(-a * (T - t))) / a


class Curve:
    """今天的曲线，以及拟合到它的 Hull-White 各量。"""

    def __init__(self, lnP, f0):
        self.lnP, self.f0 = lnP, f0

    def P(self, T):
        return np.exp(self.lnP(np.asarray(T, dtype=float)))

    def lnA(self, t, T):
        """第 31.6 节的 ln A(t,T)：正是它使模型复现这条曲线。"""
        return (self.lnP(T) - self.lnP(t) + B_hw(t, T) * self.f0(t)
                - SIG_HW ** 2 / (4 * A_HW) * (1 - np.exp(-2 * A_HW * t)) * B_hw(t, T) ** 2)

    def P_fwd(self, t, T, r):
        """式 (31.2)：短期利率为 r 时 t 时刻的债券价格。"""
        return np.exp(self.lnA(t, T) - B_hw(t, T) * r)

    def theta(self, t, h=1e-6):
        """第 31.6 节的 theta(t) = f_T(0,t) + a f(0,t) + sigma^2 (1 - e^{-2at})/(2a)。"""
        return ((self.f0(t + h) - self.f0(t - h)) / (2 * h) + A_HW * self.f0(t)
                + SIG_HW ** 2 / (2 * A_HW) * (1 - np.exp(-2 * A_HW * t)))


FLAT = Curve(lambda T: -0.04 * np.asarray(T, dtype=float), lambda t: 0.04 + 0.0 * t)

# 第 31.7 节起用的市场曲线是 R(T) = 4.2% - 1.2% exp(-0.25T)，
# 其瞬时远期利率为 f(0,T) = R(T) + T R'(T)。
MKT = Curve(lambda T: -(0.042 - 0.012 * np.exp(-0.25 * np.asarray(T, dtype=float)))
            * np.asarray(T, dtype=float),
            lambda t: 0.042 - 0.012 * np.exp(-0.25 * t) + 0.003 * t * np.exp(-0.25 * t))

print("%-10s %14s" % ("maturity", "P(0,T)"))
for T in (1, 2, 3, 5, 7, 10):
    print("%9dy %14.8f" % (T, MKT.P(T)))
show("f(0,0), which is r(0), %", 100 * float(MKT.f0(0.0)), 3.0, tol=1e-4)


def sigma_P(T, S, a=A_HW, sigma=SIG_HW):
    """式 (31.8)：远期债券价格的累积波动率。"""
    return sigma / a * (1 - exp(-a * (S - T))) * sqrt((1 - exp(-2 * a * T)) / (2 * a))


def zbp(curve, T, S, X):
    """式 (31.10)：以 S 期债券为标的、T 到期、执行价 X 的欧式看跌期权。"""
    sp = sigma_P(T, S)
    h = np.log(curve.P(S) / (X * curve.P(T))) / sp + sp / 2
    return X * curve.P(T) * N(sp - h) - curve.P(S) * N(-h)

maturity           P(0,T)
        1y     0.96787301
        2y     0.93291306
        3y     0.89673508
        5y     0.82463888
        7y     0.75623509
       10y     0.66355084
f(0,0), which is r(0), %                             3.000000   book      3.000000   ok


## 5. 上限期权的闭式解：例 31.4

平坦曲线上一份两年期、半年付、执行价 4\% 的上限期权。每一个上限单元都是
$L(1 + \tau K)$ 份以该期末债券为标的、执行价 $1/(1+\tau K)$ 的看跌期权，
而每一份看跌期权就是式 (31.10)。

尽管每一期的远期利率完全相同，上限单元的价值仍随到期期限上升，因为 $\sigma_P$
随期权的存续期增长，但这种增长是有界的，这正是均值回复在起作用。

In [8]:
TAU_CAP, K_CAP = 0.5, 0.04
X_CAP = 1.0 / (1.0 + TAU_CAP * K_CAP)
show("forward rate each period, %",
     100 * float(FLAT.P(0.5) / FLAT.P(1.0) - 1.0) / TAU_CAP, 4.040268)

print()
print("%-14s %10s %13s" % ("caplet", "sigma_P", "value"))
cap = 0.0
for T, S in ((0.5, 1.0), (1.0, 1.5), (1.5, 2.0)):
    v = L * (1.0 + TAU_CAP * K_CAP) * float(zbp(FLAT, T, S, X_CAP))
    cap += v
    print("[%.1f, %.1f]      %10.2f %13.6f" % (T, S, 10000 * sigma_P(T, S), v))
show("two-year cap", cap, 0.555017)

forward rate each period, %                          4.040268   book      4.040268   ok

caplet            sigma_P         value
[0.5, 1.0]           33.98      0.142743
[1.0, 1.5]           47.12      0.190218
[1.5, 2.0]           56.60      0.222055
two-year cap                                         0.555017   book      0.555017   ok


### 以及一份互换期权，用 Jamshidian 分解

一旦找到使附息债券恰好等于执行价的临界利率 $r^{*}$，
以附息债券为标的的欧式期权就是一组以零息债券为标的的期权的组合。
几乎全部价值都落在最后一段，因为它带着本金。

In [9]:
T_EXP, PAY = 1.0, (2.0, 3.0, 4.0)
ann = sum(float(FLAT.P(t)) for t in PAY)
par = float(FLAT.P(T_EXP) - FLAT.P(PAY[-1])) / ann
show("annuity", ann, 2.662181)
show("par swap rate, %", 100 * par, 4.081077)

COUPONS = [par, par, par + 1.0]


def coupon_bond(r):
    """标的互换的固定端，从到期日看过去。"""
    return sum(c * FLAT.P_fwd(T_EXP, t, r) for c, t in zip(COUPONS, PAY))


r_star = brentq(lambda r: coupon_bond(r) - 1.0, -0.5, 0.5, xtol=1e-14)
show("r* solving the Jamshidian threshold, %", 100 * r_star, 3.987895)

print()
print("%-10s %11s %11s %13s" % ("maturity", "c_i", "X_i", "value"))
swaption = 0.0
for c, t in zip(COUPONS, PAY):
    X_i = float(FLAT.P_fwd(T_EXP, t, r_star))
    v = L * c * float(zbp(FLAT, T_EXP, t, X_i))
    swaption += v
    print("%9.0fy %11.6f %11.6f %13.6f" % (t, c, X_i, v))
show("one-into-three payer swaption", swaption, 0.946713)

annuity                                              2.662181   book      2.662181   ok
par swap rate, %                                     4.081077   book      4.081077   ok
r* solving the Jamshidian threshold, %               3.987895   book      3.987895   ok

maturity           c_i         X_i         value
        2y    0.040811    0.960860      0.014025
        3y    0.040811    0.923177      0.025775
        4y    1.040811    0.886915      0.906913
one-into-three payer swaption                        0.946713   book      0.946713   ok


### 两者再算一遍，用积分

这是一项独立的校验。把收益直接对 $r(T)$ 的分布积分，除了债券价格本身之外，
与 Jamshidian 分解不共用任何公式，而且它只在 $T$ 远期测度下才吻合，
因为在那里 $r(T)$ 的均值恰好是 $f(0,T)$。在风险中性测度下，均值还要多带一项
$(\sigma^2/(2a^2))(1-e^{-aT})^2$，最后一行显示的就是这一次失误的代价。

In [10]:
def integrate(T, payoff, risk_neutral=False, n=40001):
    """P(0,T) E^T[收益(r(T))]。在 Q^T 下 r(T) 的均值恰好是 f(0,T)。"""
    mu = 0.04 + (SIG_HW ** 2 / (2 * A_HW ** 2) * (1 - exp(-A_HW * T)) ** 2
                 if risk_neutral else 0.0)
    sd = sqrt(SIG_HW ** 2 / (2 * A_HW) * (1 - exp(-2 * A_HW * T)))
    r = np.linspace(mu - 9 * sd, mu + 9 * sd, n)
    w = np.ones(n)
    w[1:-1:2], w[2:-1:2] = 4.0, 2.0
    dens = np.exp(-0.5 * ((r - mu) / sd) ** 2) / (sd * sqrt(2 * pi))
    return float(FLAT.P(T)) * float(np.sum(w * payoff(r) * dens)) * (r[1] - r[0]) / 3.0


direct = integrate(T_EXP, lambda r: L * np.maximum(1.0 - coupon_bond(r), 0.0))
show("the swaption again, integrated under Q^T", direct, 0.946713)
print("    两条路径相差 %.2e" % abs(swaption - direct))

wrong = integrate(T_EXP, lambda r: L * np.maximum(1.0 - coupon_bond(r), 0.0),
                  risk_neutral=True)
show("the same integral under Q instead", wrong, 0.952366)
show("what that one measure slip costs, %", 100 * (wrong / direct - 1.0), 0.6, tol=5e-2)

the swaption again, integrated under Q^T             0.946713   book      0.946713   ok
    两条路径相差 1.00e-09
the same integral under Q instead                    0.952366   book      0.952366   ok
what that one measure slip costs, %                  0.597170   book      0.600000   ok


## 6. 同一份上限单元的三种算法：例 31.5

市场曲线上一份以 $[2.0, 2.5]$ 区间单利利率为标的、执行价 4\% 的上限单元。

**(a) 在远期测度下**。远期利率是一个鞅，因此没有漂移需要建模，
而这份上限单元就是 $L(1+\tau K)$ 份以 2.5 年期债券为标的的看跌期权。
那只债券在这里服从对数正态分布，且波动率是确定性的，这恰恰是布莱克模型自己的设定，
所以这个闭式解是精确的而不是近似。两条通往它的路径都写在下面。

In [11]:
T1, T2, K = 2.0, 2.5, 0.04
TAU = T2 - T1
P_T1, P_T2 = float(MKT.P(T1)), float(MKT.P(T2))
F0 = (P_T1 / P_T2 - 1.0) / TAU
show("forward rate over [2.0, 2.5], %", 100 * F0, 3.9381, tol=2e-4)

sp = sigma_P(T1, T2)
show("forward bond volatility, bp", 10000 * sp, 64.12, tol=1e-4)
d1 = log((1 + TAU * F0) / (1 + TAU * K)) / sp + sp / 2
caplet = L * P_T2 * ((1 + TAU * F0) * N(d1) - (1 + TAU * K) * N(d1 - sp))
show("(a) the caplet under the forward measure", caplet, 0.22478353)

X_B = 1.0 / (1.0 + TAU * K)
h = log(P_T2 / (X_B * P_T1)) / sp + sp / 2
show("(a) again, as L(1 + tau K) bond puts",
     L * (1 + TAU * K) * (X_B * P_T1 * N(sp - h) - P_T2 * N(-h)), 0.22478353)

forward rate over [2.0, 2.5], %                      3.938048   book      3.938100   ok
forward bond volatility, bp                         64.122465   book     64.120000   ok
(a) the caplet under the forward measure             0.224784   book      0.224784   ok
(a) again, as L(1 + tau K) bond puts                 0.224784   book      0.224784   ok


### (b) 在风险中性测度下

现在计价单位是货币市场账户，因此收益必须沿着 $r$ 的每一条路径贴现。
这里不需要按时间步进：在 $Q$ 下，$\left(r(T_1), \int_0^{T_1} r\,ds\right)$
是联合高斯的，其各阶矩都有闭式解，因此抽样是精确的，带标准误差、但不带离散化偏差。

打印出来的第一行和答案一样有价值。$e^{-I}$ 的均值必须回到 $P(0,2)$：
正是这个恒等式定住了 $E^{Q}[I]$，而它说明滚动的货币市场账户的期望值不是 $1/P(0,T)$，
而是要带一个方差项。

书中的 0.22498411 来自两百万条对偶路径、每条两百个时间步的模拟；这里的抽样在时间上是精确的，
因此它与 (a) 的吻合程度以其自身的标准误差衡量，而不会复现书中的各位数字。

In [12]:
# 在 Q 下，r(T1) 与 I = int_0^T1 r ds 都服从高斯分布，且其联合分布已知，
# 因此这次模拟不需要时间网格，也不带任何离散化偏差。
var_r = SIG_HW ** 2 * (1 - exp(-2 * A_HW * T1)) / (2 * A_HW)
var_I = (SIG_HW / A_HW) ** 2 * (T1 - 2 * (1 - exp(-A_HW * T1)) / A_HW
                                + (1 - exp(-2 * A_HW * T1)) / (2 * A_HW))
cov_rI = (SIG_HW ** 2 / A_HW) * ((1 - exp(-A_HW * T1)) / A_HW
                                 - (1 - exp(-2 * A_HW * T1)) / (2 * A_HW))
mean_r = float(MKT.f0(T1)) + SIG_HW ** 2 / (2 * A_HW ** 2) * (1 - exp(-A_HW * T1)) ** 2
mean_I = -float(MKT.lnP(T1)) + var_I / 2      # 这就是 E[e^{-I}] = P(0,T1) 移项后的形式

rng = np.random.default_rng(31)
z = rng.standard_normal((1_000_000, 2))
z = np.concatenate([z, -z])                   # 对偶变量，因此总共两百万条
r_T1 = mean_r + sqrt(var_r) * z[:, 0]
I = mean_I + cov_rI / sqrt(var_r) * z[:, 0] + sqrt(var_I - cov_rI ** 2 / var_r) * z[:, 1]

show("E^Q[e^{-I}], which must be P(0,2)", float(np.exp(-I).mean()), P_T1, tol=1e-5)
bond = MKT.P_fwd(T1, T2, r_T1)
payoff = L * np.maximum(1.0 / bond - (1.0 + TAU * K), 0.0)
pv = np.exp(-I) * bond * payoff
se = pv.std(ddof=1) / sqrt(pv.size)
show("(b) the same caplet, simulated under Q", float(pv.mean()), caplet, tol=5e-3)
print("    标准误差 %.8f，因此这个差距是它的 %.2f 倍"
      % (se, abs(pv.mean() - caplet) / se))

E^Q[e^{-I}], which must be P(0,2)                    0.932913   book      0.932913   ok


(b) the same caplet, simulated under Q               0.224877   book      0.224784   ok
    标准误差 0.00023569，因此这个差距是它的 0.40 倍


### (c) 测度变换所要防止的误差

在 $Q$ 下计算期望收益，再用今天的零息利率贴现它，即把收益与贴现因子当作独立的。
在 $Q$ 下远期债券比率不是鞅，因此这一步也有闭式解：它就是 (a) 自己的公式，
只是把远期值乘上 $e^{\mu}$，其中 $\mu$ 正是换计价单位所消去的漂移。

这个误差就是收益与贴现因子之间的协方差。一份上限单元在利率高时支付，
而利率高时贴现因子小，因此忽略这一相关性会高估价值。

In [13]:
# (c) 用今天的零息利率贴现 E^Q[收益]。在 Q 下远期债券比率不是鞅：
# 它的漂移是 int_0^T1 nu(t) sigma_B(t,T2) dt，其中 nu 是该比率自身的波动率，
# 于是 (a) 的公式仍然适用，只需把 (1 + tau F0) 换成 (1 + tau F0)e^{mu}。
mu_Q = ((SIG_HW / A_HW) ** 2 * (1 - exp(-A_HW * TAU))
        * ((1 - exp(-A_HW * T1)) / A_HW
           - exp(-A_HW * TAU) * (1 - exp(-2 * A_HW * T1)) / (2 * A_HW)))
fwd_Q = (1 + TAU * F0) * exp(mu_Q)
d1c = log(fwd_Q / (1 + TAU * K)) / sp + sp / 2
naive = L * P_T2 * (fwd_Q * N(d1c) - (1 + TAU * K) * N(d1c - sp))
show("(c) E^Q[payoff] discounted at today's rate", naive, 0.23044665)
show("(c) the same, from the paths above", float(payoff.mean()) * P_T2, naive, tol=5e-3)
show("(c) error from the dropped covariance, %", 100 * (naive / caplet - 1.0), 2.52, tol=4e-3)

(c) E^Q[payoff] discounted at today's rate           0.230447   book      0.230447   ok
(c) the same, from the paths above                   0.230544   book      0.230447   ok
(c) error from the dropped covariance, %             2.519365   book      2.520000   ok


## 7. 远期价格不是 $Q$ 下的期望：例 31.6

远期价格在 $T$ 远期测度下是期望，而在风险中性测度下不是。
这个差距有两部分：一部分在 $\rho = 0$ 处仍然存在，来自贴现因子自身的凸性，
即 $e^{\mathrm{Var}^{Q}(\int_0^T r\,ds)}$，也就是刚刚出现过的方差项；另一部分随资产与利率之间的相关性变化。两部分合起来就是该资产与贴现因子之间的协方差。

In [14]:
S0, SIG_S, T_EQ = 100.0, 0.25, 5.0
P_5 = float(MKT.P(T_EQ))
show("P(0,5) from the curve", P_5, 0.824639, tol=1e-5)
show("the forward price S0/P(0,T)", S0 / P_5, 121.2652, tol=1e-5)

B_T = float(B_hw(0.0, T_EQ))
cov_unit = (SIG_HW / A_HW) * (T_EQ - B_T)
var_Y = (SIG_HW / A_HW) ** 2 * (T_EQ - 2 * B_T + (1 - exp(-2 * A_HW * T_EQ)) / (2 * A_HW))
print()
print("%8s %14s %10s     %s" % ("rho", "E^Q[S_T]/F", "gap", "book"))
for rho, want in ((-0.50, -1.056), (-0.25, -0.374), (0.00, 0.313),
                  (0.25, 1.004), (0.50, 1.700)):
    ratio = exp(var_Y + SIG_S * rho * cov_unit)
    flag = "ok" if abs(100 * (ratio - 1) - want) < 6e-3 else "DIFFERS"
    print("%+8.2f %14.6f %9.3f%%     %s" % (rho, ratio, 100 * (ratio - 1), flag))
show("at rho = 0, the convexity factor alone", exp(var_Y), 1.003125)

P(0,5) from the curve                                0.824639   book      0.824639   ok
the forward price S0/P(0,T)                        121.265202   book    121.265200   ok

     rho     E^Q[S_T]/F        gap     book
   -0.50       0.989442    -1.056%     ok
   -0.25       0.996260    -0.374%     ok
   +0.00       1.003125     0.313%     ok
   +0.25       1.010037     1.004%     ok
   +0.50       1.016997     1.700%     ok
at rho = 0, the convexity factor alone               1.003125   book      1.003125   ok


## 8. 一步一步把树建起来：例 31.7

第一步为去掉 $\theta(t)$、向零回复的过程 $dx = -ax\,dt + \sigma\,dW$ 建树。
间距 $\Delta x = \sigma\sqrt{3\Delta t}$ 使三个分支概率在中心处恰好是
$1/6$、$2/3$、$1/6$，而中间的概率在 $a j \Delta t > \sqrt{2/3}$ 之后变为负数，
$j_{\max}$ 由此而来，外层节点的分支方式也因此不同。

注意 $j = \pm 3$ 处那两行切换过的概率：它们朝中心而不是朝外分支，
树就是这样拒绝继续变宽的。

In [15]:
A0, SIG0 = 0.08, 0.01


def branch(j, a, dt, jmax):
    """第 j 层上的三个概率，以及它们分支到的层。"""
    u = a * j * dt
    if j == jmax:
        return ((7.0 / 6 + (u * u - 3 * u) / 2, -1.0 / 3 - u * u + 2 * u,
                 1.0 / 6 + (u * u - u) / 2), (0, -1, -2))
    if j == -jmax:
        return ((1.0 / 6 + (u * u + u) / 2, -1.0 / 3 - u * u - 2 * u,
                 7.0 / 6 + (u * u + 3 * u) / 2), (2, 1, 0))
    return ((1.0 / 6 + (u * u - u) / 2, 2.0 / 3 - u * u,
             1.0 / 6 + (u * u + u) / 2), (1, 0, -1))


def build(a, sigma, dt, n):
    """第一阶段是几何结构；第二阶段用阿罗-德布鲁价格解出 alpha(i)。"""
    dx = sigma * sqrt(3.0 * dt)
    jmax = int(ceil((1 - sqrt(2.0 / 3.0)) / (a * dt)))
    pr = {j: branch(j, a, dt, jmax) for j in range(-jmax, jmax + 1)}
    Q, alpha = [{0: 1.0}], []
    for i in range(n):
        num = sum(q * exp(-j * dx * dt) for j, q in Q[i].items())
        a_i = (log(num) - float(MKT.lnP((i + 1) * dt))) / dt
        alpha.append(a_i)
        nxt = {}
        for j, q in Q[i].items():
            (pu, pm, pd), off = pr[j]
            d = q * exp(-(a_i + j * dx) * dt)
            for p, o in zip((pu, pm, pd), off):
                nxt[j + o] = nxt.get(j + o, 0.0) + p * d
        Q.append(nxt)
    return dx, pr, alpha, Q


dx1, pr1, al1, Q1 = build(A0, SIG0, 1.0, 4)
jmax1 = max(pr1)
show("dx = sigma sqrt(3 dt)", dx1, 0.017321)
show("j_max, the smallest integer >= 0.1835/(a dt)", float(jmax1), 3.0, tol=1e-9)

print()
print("%5s %10s %10s %10s   %s" % ("j", "p_u", "p_m", "p_d", "branches to"))
for j in range(jmax1, -jmax1 - 1, -1):
    (pu, pm, pd), off = pr1[j]
    print("%+5d %10.4f %10.4f %10.4f   %+d, %+d, %+d"
          % (j, pu, pm, pd, j + off[0], j + off[1], j + off[2]))

dx = sigma sqrt(3 dt)                                0.017321   book      0.017321   ok
j_max, the smallest integer >= 0.1835/(a dt)         3.000000   book      3.000000   ok

    j        p_u        p_m        p_d   branches to
   +3     0.8355     0.0891     0.0755   +3, +2, +1
   +2     0.0995     0.6411     0.2595   +3, +2, +1
   +1     0.1299     0.6603     0.2099   +2, +1, +0
   +0     0.1667     0.6667     0.1667   +1, +0, -1
   -1     0.2099     0.6603     0.1299   +0, -1, -2
   -2     0.2595     0.6411     0.0995   -1, -2, -3
   -3     0.0755     0.0891     0.8355   -1, -2, -3


### 第二步：阿罗–德布鲁向前递推

$Q(i,j)$ 是“在节点 $(i,j)$ 支付一个单位、在别处不支付”这件事今天的价值，
也就是第 13.3 节的状态价格在这个格点上的形式。于是位移 $\alpha(i)$ 是一个一元方程，
其右端各项都已算出，因此曲线被还原到机器精度，而不是还原到某个容差。

输出里有三点值得注意：这些 $Q$ 并不对称，因为即便概率互为镜像，各节点上的贴现仍不同；
$\alpha$ 沿曲线上升，吸收着远期利率向上的斜率；而第 3 层最下面的节点带的是负利率。

In [16]:
print("%5s" % "j", "".join("%10s" % ("i = %d" % i) for i in range(5)))
for j in range(jmax1, -jmax1 - 1, -1):
    print("%+5d" % j, "".join("%10.4f" % Q1[i][j] if j in Q1[i] else "%10s" % "---"
                              for i in range(5)))
print("%5s" % "alpha", "".join("%9.4f%%" % (100 * al1[i]) if i < len(al1)
                               else "%10s" % "---" for i in range(5)))
print()
print("%5s" % "j", "".join("%10s" % ("i = %d" % i) for i in range(4)))
for j in range(jmax1, -jmax1 - 1, -1):
    print("%+5d" % j, "".join("%9.4f%%" % (100 * (al1[i] + j * dx1)) if j in Q1[i]
                              else "%10s" % "---" for i in range(4)))
print("短期利率 r(i,j) = alpha(i) + j dx")
print()
for i in range(4):
    got = sum(Q1[i][j] * exp(-(al1[i] + j * dx1)) for j in Q1[i])
    show("level %d returns P(0,%dy)" % (i, i + 1), got, float(MKT.P(i + 1.0)), tol=1e-12)

    j      i = 0     i = 1     i = 2     i = 3     i = 4
   +3        ---       ---       ---    0.0018    0.0048
   +2        ---       ---    0.0198    0.0369    0.0477
   +1        ---    0.1613    0.2045    0.2092    0.2017
   +0     1.0000    0.6452    0.4799    0.3907    0.3356
   -1        ---    0.1613    0.2081    0.2165    0.2123
   -2        ---       ---    0.0205    0.0395    0.0528
   -3        ---       ---       ---    0.0020    0.0056
alpha    3.2654%   3.6839%   3.9736%   4.1730%       ---

    j      i = 0     i = 1     i = 2     i = 3
   +3        ---       ---       ---   9.3691%
   +2        ---       ---   7.4377%   7.6371%
   +1        ---   5.4159%   5.7056%   5.9050%
   +0    3.2654%   3.6839%   3.9736%   4.1730%
   -1        ---   1.9518%   2.2415%   2.4409%
   -2        ---       ---   0.5095%   0.7089%
   -3        ---       ---       ---  -1.0232%
短期利率 r(i,j) = alpha(i) + j dx

level 0 returns P(0,1y)                              0.967873   book      0.967

### 季度步长的树

同样的构造，在 $\Delta t = 0.25$ 上跨越十一年，即第 31.8 节报告的那棵树。
随后各期权都以每年 24 步（`SPY = 24`）定价，这棵树同样精确地复现曲线。

In [17]:
SPY = 24
dx4, pr4, al4, Q4 = build(A0, SIG0, 0.25, 44)
show("the quarter-year tree: dx", dx4, 0.008660)
show("the quarter-year tree: j_max", float(max(pr4)), 10.0, tol=1e-9)
worst = max(abs(sum(Q4[i].values()) - float(MKT.P(i * 0.25))) for i in range(45))
print("全部 44 层中的最大误差：%.3e" % worst)
print("这是机器精度而不是收敛：alpha(i) 是精确解出一个方程得到的")

the quarter-year tree: dx                            0.008660   book      0.008660   ok
the quarter-year tree: j_max                        10.000000   book     10.000000   ok
全部 44 层中的最大误差：1.110e-16
这是机器精度而不是收敛：alpha(i) 是精确解出一个方程得到的


## 9. 百慕大互换期权：例 31.8

先拿一个闭式解检验这套方法。一份一年后到期的欧式收取方期权按 Jamshidian 分解值 1.1465，
树应当收敛到它。

它确实收敛，但*不是*单调的。每年 8 步比 4 步更差。
第 20.2 节已经说明为什么：格点的误差取决于节点相对于收益折点的落位，而加密网格会移动它们。
交易台若不检查这一点就对两个树值做外推，可能得到比两者都差的答案。

In [18]:
PAYS = (2.0, 3.0, 4.0, 5.0, 6.0)


def receiver(a, sigma, spy, exercises, K=0.04, notional=100.0, years=11.0):
    """标的为 4% 固定、到期于 t = 6 的互换的收取方期权，在树上向后回溯。

    在 t 行权的互换期权买到的是严格在 t 之后的那些支付，
    因此行权判断要放在加上当日票息之前。"""
    dt = 1.0 / spy
    n = int(round(years / dt))
    dx, pr, alpha, Q = build(a, sigma, dt, n)
    val = {j: 0.0 for j in Q[n]}
    swap = {j: 0.0 for j in Q[n]}
    for i in range(n - 1, -1, -1):
        nv, ns = {}, {}
        for j in Q[i]:
            (pu, pm, pd), off = pr[j]
            d = exp(-(alpha[i] + j * dx) * dt)
            nv[j] = d * sum(p * val.get(j + o, 0.0) for p, o in zip((pu, pm, pd), off))
            ns[j] = d * sum(p * swap.get(j + o, 0.0) for p, o in zip((pu, pm, pd), off))
        t = i * dt
        if any(abs(t - e) < dt / 2 for e in exercises):
            nv = {j: max(nv[j], notional * (ns[j] - 1.0)) for j in nv}
        for tp in PAYS:
            if abs(t - tp) < dt / 2:
                ns = {j: v + K + (1.0 if tp == PAYS[-1] else 0.0) for j, v in ns.items()}
        val, swap = nv, ns
    return val[0]


print("%12s %12s %12s   %s" % ("steps/year", "tree", "error", "book"))
for spy, want in ((4, 1.1637), (8, 1.1696), (12, 1.1667), (24, 1.1586), (48, 1.1502)):
    v = receiver(A0, SIG0, spy, (1.0,))
    flag = "ok" if abs(v - want) < 5e-5 else "DIFFERS"
    print("%12d %12.4f %+12.4f   %s" % (spy, v, v - 1.1465, flag))
print("一年期欧式期权按 Jamshidian 分解的闭式值是 1.1465")

  steps/year         tree        error   book
           4       1.1637      +0.0172   ok
           8       1.1696      +0.0231   ok
          12       1.1667      +0.0202   ok


          24       1.1586      +0.0121   ok


          48       1.1502      +0.0037   ok
一年期欧式期权按 Jamshidian 分解的闭式值是 1.1465


### 一份有五个日期的期权，而不是五份期权

现在看产品本身。百慕大比任何单份欧式都贵，因为选择的权利有价值；
又远低于它们之和，因为这些权利是互相替代的。
五份欧式的组合是五份期权；一份百慕大是一份有五个日期的期权。

In [19]:
EX = (1.0, 2.0, 3.0, 4.0, 5.0)
europeans = [receiver(A0, SIG0, SPY, (e,)) for e in EX]
bermudan = receiver(A0, SIG0, SPY, EX)
print("%-30s %10s" % ("exercise right", "value"))
for e, v in zip(EX, europeans):
    print("%-30s %10.4f" % ("European, %dy only" % e, v))
print("%-30s %10.4f" % ("sum of all five", sum(europeans)))
print("%-30s %10.4f" % ("Bermudan, any one of the five", bermudan))
print()
show("over the best single European, %", 100 * (bermudan / max(europeans) - 1), 42.2, tol=5e-3)
show("as a share of their sum, %", 100 * bermudan / sum(europeans), 37.0, tol=2e-2)

exercise right                      value
European, 1y only                  1.1586
European, 2y only                  1.1953
European, 3y only                  1.0559
European, 4y only                  0.7974
European, 5y only                  0.4412
sum of all five                    4.6484
Bermudan, any one of the five      1.6995

over the best single European, %                    42.181087   book     42.200000   ok
as a share of their sum, %                          36.560480   book     37.000000   ok


## 10. 每一份欧式都匹配，百慕大不匹配：例 31.9

把用于校准的欧式期权固定在它的树上价值，对每一个均值回复速度解出精确复现它的 $\sigma$。
每一行给校准工具的定价在小数点后四位完全相同，而百慕大价格的变动范围达到其自身价值的 13\%。

单独一份欧式**只观察一次**曲线，因此它不含关于 $a$ 的任何信息；
而 $a$ 支配的正是“第三年的曲线包含多少关于第四年曲线的信息”，
百慕大期权的价值恰恰依赖于此。

这个单元格要为五个均值回复速度各求一次 $\sigma$，每一步都重建一次树，
因此它是整个笔记本里最慢的一格。

In [20]:
target = europeans[0]
print("%6s %10s %12s %12s %10s   %s"
      % ("a", "sigma", "European", "Bermudan", "vs 0.08", "book"))
values = []
for a, want_s, want_b in ((0.01, 0.00823, 1.6474), (0.03, 0.00871, 1.6622),
                          (0.08, 0.01000, 1.6995), (0.15, 0.01202, 1.7541),
                          (0.30, 0.01724, 1.8705)):
    sg = brentq(lambda s: receiver(a, s, SPY, (1.0,)) - target, 0.0005, 0.06, xtol=1e-9)
    b = receiver(a, sg, SPY, EX)
    values.append(b)
    flag = "ok" if abs(sg - want_s) < 5e-6 and abs(b - want_b) < 5e-5 else "DIFFERS"
    print("%6.2f %10.5f %12.4f %12.4f %+9.1f%%   %s"
          % (a, sg, receiver(a, sg, SPY, (1.0,)), b, 100 * (b / bermudan - 1), flag))
print()
show("the Bermudan's span, % of its own value",
     100 * (max(values) - min(values)) / values[2], 13.1, tol=5e-3)

     a      sigma     European     Bermudan    vs 0.08   book


  0.01    0.00823       1.1586       1.6474      -3.1%   ok


  0.03    0.00871       1.1586       1.6622      -2.2%   ok


  0.08    0.01000       1.1586       1.6995      -0.0%   ok


  0.15    0.01202       1.1586       1.7541      +3.2%   ok


  0.30    0.01724       1.1586       1.8705     +10.1%   ok

the Bermudan's span, % of its own value             13.125662   book     13.100000   ok


## 11. Z 利差与期权调整利差：例 31.10

一只十年期、票息 5\%、自第三年起可按面值赎回的债券，价格 99。
Z 利差是忽略赎回权时使它定到这个价的平行移动；OAS 是把赎回权建模之后使它定到这个价的移动。
两者之差是赎回权在债券自身利差下的价值，折算成每年的收益率。

第二张表给出的是警告。参数取值若未经共同校准确定，OAS 可以从 21 个基点走到 72 个基点，
因为它是两个几乎相等的量之*差*，
期权价值的小幅变化就会造成残差的大幅变化。Z 利差是关于价格的事实；
OAS 是关于价格**以及**一个模型的事实。

In [21]:
def callable_bond(a, sigma, spread, callable_=True, cpn=0.05, mat=10, first=3, years=11.0):
    """一只十年期、票息 5%、自第三年起每年可按面值赎回的债券。"""
    dt = 1.0 / SPY
    n = int(round(years / dt))
    dx, pr, alpha, Q = build(a, sigma, dt, n)
    v = {j: 0.0 for j in Q[n]}
    for i in range(n - 1, -1, -1):
        nv = {}
        for j in Q[i]:
            (pu, pm, pd), off = pr[j]
            d = exp(-(alpha[i] + j * dx + spread) * dt)
            nv[j] = d * sum(p * v.get(j + o, 0.0) for p, o in zip((pu, pm, pd), off))
        t = i * dt
        if callable_ and any(abs(t - c) < dt / 2 for c in range(first, mat)):
            nv = {j: min(nv[j], 100.0) for j in nv}
        for tp in range(1, mat + 1):
            if abs(t - tp) < dt / 2:
                nv = {j: x + 100.0 * cpn + (100.0 if tp == mat else 0.0)
                      for j, x in nv.items()}
        v = nv
    return v[0]


straight = callable_bond(A0, SIG0, 0.0, callable_=False)
called = callable_bond(A0, SIG0, 0.0, callable_=True)
show("the same bond with the call switched off", straight, 106.9013, tol=5e-6)
show("and with it switched on", called, 101.6743, tol=5e-6)
show("so the embedded call is worth", straight - called, 5.2270, tol=5e-5)
print()

z_sp = brentq(lambda s: callable_bond(A0, SIG0, s, callable_=False) - 99.0,
              -0.02, 0.2, xtol=1e-10)
oas = brentq(lambda s: callable_bond(A0, SIG0, s, callable_=True) - 99.0,
             -0.02, 0.2, xtol=1e-10)
show("Z-spread, bp", 10000 * z_sp, 94.6, tol=5e-4)
show("option-adjusted spread, bp", 10000 * oas, 52.1, tol=5e-4)
show("difference, bp", 10000 * (z_sp - oas), 42.5, tol=2e-3)   # 书里相减的是两个已经取整的数

print()
print("%8s %10s %12s   %s" % ("a", "sigma", "OAS (bp)", "book"))
for a, sg, want in ((0.03, 0.010, 39.5), (0.08, 0.010, 52.1),
                    (0.20, 0.010, 71.5), (0.08, 0.015, 20.9)):
    oo = brentq(lambda s: callable_bond(a, sg, s, callable_=True) - 99.0,
                -0.02, 0.2, xtol=1e-10)
    flag = "ok" if abs(10000 * oo - want) < 5e-2 else "DIFFERS"
    print("%8.2f %10.3f %12.1f   %s" % (a, sg, 10000 * oo, flag))

the same bond with the call switched off           106.901261   book    106.901300   ok
and with it switched on                            101.674344   book    101.674300   ok
so the embedded call is worth                        5.226917   book      5.227000   ok



Z-spread, bp                                        94.560256   book     94.600000   ok
option-adjusted spread, bp                          52.106552   book     52.100000   ok
difference, bp                                      42.453704   book     42.500000   ok

       a      sigma     OAS (bp)   book


    0.03      0.010         39.5   ok


    0.08      0.010         52.1   ok


    0.20      0.010         71.5   ok


    0.08      0.015         20.9   ok


## 12. 完全相关的代价：例 31.11

一份五年期、以十年期与两年期互换利率之差为标的的期权。该利差的波动率是
$\sqrt{\sigma_{10}^2 + \sigma_2^2 - 2\rho\sigma_{10}\sigma_2}$，
而单因子模型被迫取 $\rho = 1$，也就是第一行。

在 $\rho = 1$ 处，这个利差波动率甚至不是一项相关性假定。它是
$|\sigma_{10} - \sigma_2|$，是两个利率以略微不同的速度齐步移动之后剩下的残差。
没有任何校准能修好这一点，而且误差的符号永远一样，因为 $\rho = 1$ 使利差的波动率*最小*。

In [22]:
F10, F2, S10, S2, ANN, T_CMS = 0.0340, 0.0280, 0.0090, 0.0075, 4.20, 5.0
show("the forward spread, bp", 10000 * (F10 - F2), 60.0, tol=5e-4)
show("at rho = 1 the spread vol is |sigma_10 - sigma_2|, bp",
     10000 * abs(S10 - S2), 15.0, tol=5e-4)

print()
print("%8s %14s %16s %8s   %s"
      % ("rho", "spread vol", "option value", "ratio", "book"))
base = None
for rho, w_vol, w_val in ((1.00, 15.0, 56.2), (0.95, 30.0, 112.4), (0.90, 39.7, 148.7),
                          (0.80, 54.1, 202.6), (0.70, 65.4, 245.0), (0.50, 83.5, 312.9)):
    vol = sqrt(S10 ** 2 + S2 ** 2 - 2 * rho * S10 * S2)
    value = ANN * vol * sqrt(T_CMS / (2 * pi))      # Bachelier 模型，平值执行
    base = base or value
    flag = "ok" if abs(10000 * vol - w_vol) < 5e-2 and abs(10000 * value - w_val) < 5e-2 \
        else "DIFFERS"
    print("%8.2f %11.1f bp %13.1f bp %8.2f   %s"
          % (rho, 10000 * vol, 10000 * value, value / base, flag))

the forward spread, bp                              60.000000   book     60.000000   ok
at rho = 1 the spread vol is |sigma_10 - sigma_2|, bp      15.000000   book     15.000000   ok

     rho     spread vol     option value    ratio   book
    1.00        15.0 bp          56.2 bp     1.00   ok
    0.95        30.0 bp         112.4 bp     2.00   ok
    0.90        39.7 bp         148.7 bp     2.65   ok
    0.80        54.1 bp         202.6 bp     3.61   ok
    0.70        65.4 bp         245.0 bp     4.36   ok
    0.50        83.5 bp         312.9 bp     5.57   ok


## 13. 模型说利率低于零的频率：例 31.12

高斯型短期利率以正概率变为负数，而期限为十年时，中间一行给出的概率超过五分之一。

在利率转负之前（丹麦 2012 年、欧元区 2014 年），这被当作这个模型的首要缺陷来引用。后来利率转负并持续数年，
于是这项性质成了 Hull–White 得以存续的理由，而它那些保持为正的竞争者不加位移就根本无法被校准。
模型认为不合情理的区域，正是它下一次受检验的地方。

In [23]:
print("%-26s %8s %8s %8s %8s   %s" % ("", "1y", "2y", "5y", "10y", "book"))
for a, sg, want in ((0.05, 0.010, (0.1, 1.3, 6.5, 11.6)),
                    (0.05, 0.015, (2.0, 6.9, 15.7, 21.3)),
                    (0.10, 0.010, (0.1, 1.0, 4.6, 7.5))):
    got = []
    for T in (1.0, 2.0, 5.0, 10.0):
        sd = sqrt(sg ** 2 / (2 * a) * (1 - exp(-2 * a * T)))
        got.append(100 * N(-0.03 / sd))
    flag = "ok" if max(abs(g - w) for g, w in zip(got, want)) < 5e-2 else "DIFFERS"
    print("%-26s %7.1f%% %7.1f%% %7.1f%% %7.1f%%   %s"
          % ("a = %.2f, sigma = %.1f%%" % (a, 100 * sg), *got, flag))
print("在 3% 的平坦远期曲线下，Hull-White 的 r(T) 低于零的概率")

                                 1y       2y       5y      10y   book
a = 0.05, sigma = 1.0%         0.1%     1.3%     6.5%    11.6%   ok
a = 0.05, sigma = 1.5%         2.0%     6.9%    15.7%    21.3%   ok
a = 0.10, sigma = 1.0%         0.1%     1.0%     4.6%     7.5%   ok
在 3% 的平坦远期曲线下，Hull-White 的 r(T) 低于零的概率
